# 3. Przestrzenie Hilberta, rzuty i przybliżenia

**Arkadiusz Liber** · *Falki w praktyce. Od wzoru do kodu*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/writingin/falki-w-praktyce-od-wzoru-do-kodu-notatniki/blob/main/rozdzial-03.ipynb)

Notatnik do książki. Zawiera wszystkie programy z tekstu rozdziału oraz ćwiczenia z rozwiązaniami.

Notatnik działa lokalnie (Jupyter) i w Google Colab. Komórki można uruchamiać po kolei poleceniem *Uruchom wszystko*.

In [1]:
# Uruchom tę komórkę jako pierwszą (lokalnie albo w Google Colab).
import sys, subprocess, importlib.util, os
if importlib.util.find_spec("pywt") is None:      # PyWavelets
    subprocess.check_call([sys.executable, "-m", "pip",
                           "install", "-q", "PyWavelets"])
import numpy as np
import matplotlib.pyplot as plt
import pywt

SZARY, NIEBIESKI, RDZAWY, ZIELONY = "#757575", "#1273E2", "#D93B0B", "#078832"
FIOLET, BURSZTYN = "#9258D5", "#B36000"
plt.rcParams.update({
    "text.color": SZARY, "axes.labelcolor": SZARY, "axes.edgecolor": SZARY,
    "xtick.color": SZARY, "ytick.color": SZARY, "axes.titlecolor": SZARY,
    "font.size": 10, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False})


## Programy z tekstu rozdziału

### Kod 3.1. Kresy Riesza bazy nieortogonalnej

In [2]:
import numpy as np

def kresy_riesza(V):
    """Wiersze V: wektory bazy. Zwraca C1, C2."""
    G = V @ V.conj().T          # macierz Grama
    w = np.linalg.eigvalsh(G)
    return w[0], w[-1]

dobra = np.array([[1, 0], [0.6, 0.8]])
zla = np.array([[1, 0], [0.999, 0.045]])

for nazwa, V in [("dobra", dobra), ("zła", zla)]:
    c1, c2 = kresy_riesza(V)
    print(f"baza {nazwa}: C1 = {c1:.4f}, "
          f"C2 = {c2:.4f}, C2/C1 = {c2/c1:.0f}")

baza dobra: C1 = 0.4000, C2 = 1.6000, C2/C1 = 4
baza zła: C1 = 0.0010, C2 = 1.9990, C2/C1 = 1973


### Kod 3.2. Macierz Hilberta: kresy Riesza bazy potęgowej i utrata dokładności

In [3]:
import numpy as np
import sympy as sp

x = sp.symbols("x")
for n in (3, 6, 9, 12):
    G = np.array([[1/(j + k + 1) for k in range(n)] for j in range(n)])
    w = np.linalg.eigvalsh(G)
    print(f"n = {n:2d}: kresy Riesza {w[0]:.1e} i {w[-1]:.2f}, "
          f"wskaźnik uwarunkowania {w[-1]/w[0]:.1e}")

# najlepsze przybliżenie funkcji exp(x) wielomianem stopnia 11
n = 12
G = sp.Matrix(n, n, lambda j, k: sp.Rational(1, j + k + 1))
b = sp.Matrix([sp.integrate(x**j*sp.exp(x), (x, 0, 1)) for j in range(n)])
dokladne = np.array(G.LUsolve(b).evalf(30), dtype=float).ravel()

G_f = np.array(G, dtype=float)
b_f = np.array(b.evalf(30), dtype=float).ravel()
numeryczne = np.linalg.solve(G_f, b_f)
print("największy błąd względny współczynnika:",
      f"{np.max(np.abs(numeryczne - dokladne)/np.abs(dokladne)):.1e}")

n =  3: kresy Riesza 2.7e-03 i 1.41, wskaźnik uwarunkowania 5.2e+02
n =  6: kresy Riesza 1.1e-07 i 1.62, wskaźnik uwarunkowania 1.5e+07
n =  9: kresy Riesza 3.5e-12 i 1.73, wskaźnik uwarunkowania 4.9e+11
n = 12: kresy Riesza 1.1e-16 i 1.80, wskaźnik uwarunkowania 1.7e+16
największy błąd względny współczynnika: 2.8e+04


### Kod 3.3. Rzut ortogonalny na podprzestrzeń i własności rzutu

In [4]:
import numpy as np

# podprzestrzeń W rozpięta przez kolumny macierzy X
X = np.array([[1., 1], [2, 1], [3, 1], [4, 1]])
v = np.array([1, 2, 1.5, 3])

Q, _ = np.linalg.qr(X)        # baza ortonormalna W
P = Q @ Q.T                   # macierz rzutu, wzór (3.12)

rzut = P @ v
blad = v - rzut
print(np.round(rzut, 2))      # [1.05 1.6  2.15 2.7 ]
print(np.allclose(P @ P, P))  # idempotentność
print(np.allclose(X.T @ blad, 0))   # błąd prostopadły do W
print(round(float(blad @ blad), 3)) # 0.675

[1.05 1.6  2.15 2.7 ]
True
True
0.675


### Kod 3.4. Baza biortogonalna: analiza jedną bazą, synteza drugą

In [5]:
import numpy as np

# wiersze: nieortogonalna baza syntezy w R^3
V = np.array([[1., 0, 0],
              [1, 1, 0],
              [1, 1, 1]])
# wiersze: baza biortogonalna (analizy)
Vd = np.linalg.inv(V).T

print(np.allclose(Vd @ V.T, np.eye(3)))   # wzór (3.15)

v = np.array([2., 5, 4])
c = Vd @ v                 # współczynniki <v~_j | v>
print(c)                   # [-3.  1.  4.]
print(np.allclose(c @ V, v))              # wzór (3.16)

True
[-3.  1.  4.]
True


### Kod 3.5. Kolejne przybliżenia i błąd ze wzoru (3.19)

In [6]:
import numpy as np

def baza_haara(n):
    H = [np.ones(n)]
    dl = n
    while dl > 1:
        for start in range(0, n, dl):
            h = np.zeros(n)
            h[start:start + dl//2] = 1
            h[start + dl//2:start + dl] = -1
            H.append(h)
        dl //= 2
    H = np.array(H)
    return H / np.linalg.norm(H, axis=1, keepdims=True)

H = baza_haara(8)
v = np.array([4., 6, 10, 12, 8, 6, 5, 5])
c = H @ v

for k in (1, 2, 4, 8):
    przybl = H[:k].T @ c[:k]
    blad2 = np.sum((v - przybl)**2)
    print(k, np.round(przybl, 2), round(blad2, 2),
          round(float(np.sum(c[k:]**2)), 2))

1 [7. 7. 7. 7. 7. 7. 7. 7.] 54.0 54.0
2 [8. 8. 8. 8. 6. 6. 6. 6.] 46.0 46.0
4 [ 5.  5. 11. 11.  7.  7.  5.  5.] 6.0 6.0
8 [ 4.  6. 10. 12.  8.  6.  5.  5.] 0.0 0.0


### Kod 3.6. Macierz operatora różniczkowania w bazie potęgowej

In [7]:
import sympy as sp

x = sp.symbols("x")
baza = [sp.Integer(1), x, x**2, x**3]

def wspolrzedne(w):
    return [sp.Poly(w, x).coeff_monomial(x**k) for k in range(4)]

D = sp.Matrix([wspolrzedne(sp.diff(e, x)) for e in baza]).T
print(D)

w = 5 - 2*x + 4*x**3
print((D*sp.Matrix(wspolrzedne(w))).T, "|", sp.diff(w, x))
print("jądro:", D.nullspace()[0].T, " rząd:", D.rank())
print("D^4 = 0:", D**4 == sp.zeros(4))

Matrix([[0, 1, 0, 0], [0, 0, 2, 0], [0, 0, 0, 3], [0, 0, 0, 0]])
Matrix([[-2, 0, 12, 0]]) | 12*x**2 - 2
jądro: Matrix([[1, 0, 0, 0]])  rząd: 3
D^4 = 0: True


### Kod 3.7. Najlepsze przybliżenie funkcji wykładniczej wielomianem drugiego stopnia

In [8]:
import numpy as np
import sympy as sp

x = sp.symbols("x")
f = sp.exp(x)
iloczyn = lambda u, v: sp.integrate(u*v, (x, -1, 1))

# baza ortonormalna: unormowane wielomiany Legendre'a
baza = [sp.sqrt(sp.Rational(2*n + 1, 2))*sp.legendre(n, x) for n in range(3)]
rzut = sp.expand(sum(iloczyn(e, f)*e for e in baza))
taylor = 1 + x + x**2/2
print(sp.N(rzut, 5))

t = np.linspace(-1, 1, 2001)
for nazwa, w in (("rzut", rzut), ("Taylor", taylor)):
    blad = sp.lambdify(x, f - w)(t)
    energia = sp.N(iloczyn(f - w, f - w))
    print(f"{nazwa:7s} norma błędu {float(sp.sqrt(energia)):.4f}, "
          f"największy błąd {np.abs(blad).max():.4f}")

0.53673*x**2 + 1.1036*x + 0.99629
rzut    norma błędu 0.0380, największy błąd 0.0816
Taylor  norma błędu 0.0948, największy błąd 0.2183


### Kod 3.8. Trend i wahania sezonowe wyznaczone jako rzut na podprzestrzeń modelu

In [9]:
import numpy as np

rng = np.random.default_rng(3)
t = np.arange(240)/12                       # 20 lat, dane miesięczne
y = (8.0 + 0.15*t + 9.5*np.cos(2*np.pi*(t - 0.55))
     + rng.normal(scale=1.0, size=t.size))  # model + zakłócenia

X = np.stack([np.ones_like(t), t,
              np.cos(2*np.pi*t), np.sin(2*np.pi*t)], axis=1)
wsp = np.linalg.lstsq(X, y, rcond=None)[0]
dopasowanie = X @ wsp                       # rzut na podprzestrzeń
reszta = y - dopasowanie

print("poziom, trend:", wsp[:2].round(3))
print("amplituda roczna:", round(float(np.hypot(wsp[2], wsp[3])), 2))
print("reszta ortogonalna do modelu:", np.allclose(X.T @ reszta, 0))
print("Pitagoras:", np.isclose(y @ y, dopasowanie @ dopasowanie
                               + reszta @ reszta))
print("odchylenie standardowe reszty:", round(float(reszta.std()), 2))

poziom, trend: [8.015 0.149]
amplituda roczna: 9.57
reszta ortogonalna do modelu: True
Pitagoras: True
odchylenie standardowe reszty: 1.01


### Kod 3.9. Baza funkcji daszkowych, baza do niej dualna i rzut ortogonalny

In [10]:
import numpy as np

N = 1200
x = np.linspace(0, 6, N, endpoint=False)
dx = x[1] - x[0]
daszki = np.array([np.clip(1 - np.abs(x - k), 0, None)
                   for k in range(1, 6)])        # węzły 1, ..., 5

G = daszki @ daszki.T*dx                         # macierz Grama
print(G.round(3)[0], G.round(3)[1])
dualne = np.linalg.inv(G) @ daszki               # baza biortogonalna
print(np.allclose(dualne @ daszki.T*dx, np.eye(5)))
w = np.linalg.eigvalsh(G)
print("kresy Riesza:", w[0].round(3), w[-1].round(3))

f = np.sin(np.pi*x/6)*(1 + 0.6*np.sin(1.7*x))
wsp = dualne @ f*dx                              # analiza: baza dualna
rzut = wsp @ daszki                              # synteza: daszki
interpolacja = f[np.searchsorted(x, np.arange(1, 6))] @ daszki
for nazwa, p in (("rzut ortogonalny", rzut),
                 ("interpolacja", interpolacja)):
    print(nazwa, round(float(np.linalg.norm(f - p)*np.sqrt(dx)), 4))

[0.667 0.167 0.    0.    0.   ] [0.167 0.667 0.167 0.    0.   ]
True
kresy Riesza: 0.378 0.955
rzut ortogonalny 0.1776
interpolacja 0.2284


### Kod 3.10. Normy z różnymi wykładnikami

In [11]:
import numpy as np

v = np.array([3., -4, 0, 0, 0, 0, 0, 0])   # rzadki
w = np.full(8, 5/np.sqrt(8))               # rozmyty

for nazwa, x in [("rzadki", v), ("rozmyty", w)]:
    print(nazwa,
          round(np.linalg.norm(x, 1), 3),
          round(np.linalg.norm(x, 2), 3),
          round(np.linalg.norm(x, np.inf), 3))

rzadki 7.0 5.0 4.0
rozmyty 14.142 5.0 1.768


### Kod 3.11. Charakterystyki liczbowe funkcji Gaussa

In [12]:
import numpy as np

x = np.linspace(-20, 20, 40001)
dx = x[1] - x[0]
mu, s = 1.5, 2.0
f = np.exp(-(x - mu)**2 / (2*s**2))

calka = np.sum(f) * dx
energia = np.sum(f**2) * dx
srodek = np.sum(x * f) * dx / calka
wariancja = np.sum((x - srodek)**2 * f) * dx / calka
szer = calka / f[np.argmin(abs(x))]

print(f"całka      {calka:.4f}")      # s*sqrt(2*pi)
print(f"energia    {energia:.4f}")    # s*sqrt(pi)
print(f"środek     {srodek:.4f}")     # mu
print(f"wariancja  {wariancja:.4f}")  # s**2
print(f"szerokość  {szer:.4f}")

całka      5.0133
energia    3.5449
środek     1.5000
wariancja  4.0000
szerokość  6.6415


### Kod 3.12. Średnia, mediana i środek rozstępu jako najlepsze przybliżenia w trzech normach

In [13]:
import numpy as np
from scipy.optimize import minimize_scalar

y = np.array([4.1, 3.8, 4.4, 4.0, 3.9, 4.2, 9.5])   # jeden pomiar błędny

normy = {"l2": lambda c: np.sum((y - c)**2),
         "l1": lambda c: np.sum(np.abs(y - c)),
         "l_inf": lambda c: np.max(np.abs(y - c))}
for nazwa, koszt in normy.items():
    c = minimize_scalar(koszt, bounds=(0, 10), method="bounded").x
    print(f"{nazwa:6s} {c:.3f}")
print("średnia", y.mean().round(3), " mediana", np.median(y),
      " środek rozstępu", (y.min() + y.max())/2)

l2     4.843
l1     4.100
l_inf  6.650
średnia 4.843  mediana 4.1  środek rozstępu 6.65


### Kod 3.13. Prosta dopasowana w normach z wykładnikiem 1, 2 i nieskończonym

In [14]:
import numpy as np
from scipy.optimize import linprog

rng = np.random.default_rng(2)
x = np.linspace(0, 10, 21)
y = 1.0 + 0.5*x + rng.normal(scale=0.2, size=x.size)
y[[4, 15]] += [6.0, -5.0]                      # dwa błędy grube
n = len(x)
X = np.stack([np.ones(n), x], axis=1)

l2 = np.linalg.lstsq(X, y, rcond=None)[0]

# norma 1: min sum(t) przy |y - X w| <= t; zmienne (w0, w1, t_1..t_n)
A = np.block([[X, -np.eye(n)], [-X, -np.eye(n)]])
b = np.concatenate([y, -y])
c = np.concatenate([[0, 0], np.ones(n)])
l1 = linprog(c, A_ub=A, b_ub=b,
             bounds=[(None, None)]*2 + [(0, None)]*n).x[:2]

# norma maksimum: min t przy |y - X w| <= t; zmienne (w0, w1, t)
jed = np.ones((n, 1))
A = np.block([[X, -jed], [-X, -jed]])
l_inf = linprog([0, 0, 1], A_ub=A, b_ub=b,
                bounds=[(None, None)]*2 + [(0, None)]).x[:2]

for nazwa, w in (("l1", l1), ("l2", l2), ("l_inf", l_inf)):
    print(f"{nazwa:6s} wyraz wolny {w[0]:6.3f}, nachylenie {w[1]:6.3f}")

l1     wyraz wolny  0.909, nachylenie  0.509
l2     wyraz wolny  1.828, nachylenie  0.344
l_inf  wyraz wolny  4.873, nachylenie -0.174


### Kod 3.14. Przybliżenie funkcji wykładniczej wielomianem trzeciego stopnia: norma 2 i norma maksimum

In [15]:
import numpy as np
from scipy.optimize import linprog
from numpy.polynomial import legendre, polynomial

t = np.linspace(-1, 1, 2001)
f = np.exp(t)

# norma 2: rzut na wielomiany Legendre'a stopnia 0..3
l2 = legendre.legval(t, legendre.legfit(t, f, 3))

# norma maksimum: programowanie liniowe na gęstej siatce
V = polynomial.polyvander(t, 3)
jed = np.ones((t.size, 1))
A = np.block([[V, -jed], [-V, -jed]])
b = np.concatenate([f, -f])
wynik = linprog([0, 0, 0, 0, 1], A_ub=A, b_ub=b,
                bounds=[(None, None)]*4 + [(0, None)])
minimaks = V @ wynik.x[:4]

for nazwa, p in (("norma 2", l2), ("norma maksimum", minimaks)):
    e = f - p
    ekstrema = np.sum(np.diff(np.sign(np.diff(e))) != 0) + 2
    print(f"{nazwa:15s} największy błąd {np.abs(e).max():.5f}, "
          f"średniokwadratowy {np.sqrt(np.mean(e**2)):.5f}, "
          f"ekstremów błędu: {ekstrema}")

norma 2         największy błąd 0.01114, średniokwadratowy 0.00335, ekstremów błędu: 5
norma maksimum  największy błąd 0.00553, średniokwadratowy 0.00388, ekstremów błędu: 5


### Kod 3.15. Sygnał złożony z impulsów i kosinusoid: rozwiązanie o najmniejszej normie 2 i o najmniejszej normie 1

In [16]:
import numpy as np
from scipy.fft import idct
from scipy.optimize import linprog

N = 64
kosinusy = idct(np.eye(N), norm="ortho", axis=0)   # baza kosinusowa
A = np.hstack([np.eye(N), kosinusy])               # 128 wektorów w R^64

prawdziwe = np.zeros(2*N)
prawdziwe[[10, 37]] = [2.0, -1.5]                  # dwa impulsy
prawdziwe[[N + 5, N + 12]] = [3.0, 2.0]            # dwie kosinusoidy
s = A @ prawdziwe

l2 = np.linalg.pinv(A) @ s                         # najmniejsza norma 2

# najmniejsza norma 1: c = u - v, u, v >= 0, min sum(u + v)
wynik = linprog(np.ones(4*N), A_eq=np.hstack([A, -A]), b_eq=s,
                bounds=[(0, None)]*(4*N))
l1 = wynik.x[:2*N] - wynik.x[2*N:]

for nazwa, c in (("norma 2", l2), ("norma 1", l1)):
    print(f"{nazwa}: niezerowych współczynników "
          f"{np.sum(np.abs(c) > 1e-6):3d}, odtworzenie sygnału: "
          f"{np.allclose(A @ c, s)}, zgodność z prawdziwymi: "
          f"{np.allclose(c, prawdziwe, atol=1e-6)}")

norma 2: niezerowych współczynników 128, odtworzenie sygnału: True, zgodność z prawdziwymi: False
norma 1: niezerowych współczynników   4, odtworzenie sygnału: True, zgodność z prawdziwymi: True


### Kod 3.16. Energia, środek ciężkości i szerokość funkcji Gaussa i impulsu prostokątnego w postaci wzorów

In [17]:
import sympy as sp

x, m = sp.symbols("x m", real=True)
s, a = sp.symbols("sigma a", positive=True)
oo = sp.oo

def charakterystyki(f, granice):
    energia = sp.integrate(f**2, (x, *granice))
    gestosc = f**2/energia
    srodek = sp.integrate(x*gestosc, (x, *granice))
    wariancja = sp.integrate((x - srodek)**2*gestosc, (x, *granice))
    return [sp.simplify(w) for w in (energia, srodek, sp.sqrt(wariancja))]

gauss = sp.exp(-(x - m)**2/(2*s**2))
print(charakterystyki(gauss, (-oo, oo)))
print(charakterystyki(sp.Integer(1), (m - a/2, m + a/2)))

[sqrt(pi)*sigma, m, sqrt(2)*sigma/2]
[a, m, sqrt(3)*a/6]


## Ćwiczenia i rozwiązania

### Ćwiczenie 3.1
Dla bazy (1, 0), (1, 1) wyznacz bazę biortogonalną i kresy Riesza.

### Rozwiązanie 3.1

In [18]:
V = np.array([[1., 0], [1, 1]])
print(np.linalg.inv(V).T)              # wiersze: baza biortogonalna
print(np.linalg.eigvalsh(V @ V.T))     # kresy Riesza

[[ 1. -1.]
 [ 0.  1.]]
[0.38196601 2.61803399]


### Ćwiczenie 3.2
Pokaż na przykładzie wektorów (1, 0) i (0, 1), że wyrażenie (3.23) z wykładnikiem *p* = 1/2 nie spełnia nierówności trójkąta.

### Rozwiązanie 3.2

In [19]:
norma = lambda x, p: np.sum(np.abs(x)**p)**(1/p)
a, b = np.array([1., 0]), np.array([0., 1])
print(norma(a + b, 0.5), "<=?", norma(a, 0.5) + norma(b, 0.5))

4.0 <=? 2.0


### Ćwiczenie 3.3
Oblicz momenty rzędu 0, 1 i 2 falki „kapelusz meksykański" ψ(x) = (1 − x²) exp(−x²/2). Ile momentów zerowych ma ta falka?

### Rozwiązanie 3.3

In [20]:
x = np.linspace(-20, 20, 400001); dx = x[1] - x[0]
psi = (1 - x**2) * np.exp(-x**2 / 2)
for k in range(3):
    print(k, round(float(np.sum(x**k * psi) * dx), 6))
# Momenty rzędu 0 i 1 są zerami, moment rzędu 2 nie: falka ma dwa momenty zerowe.

0 -0.0
1 0.0
2 -5.013257


### Ćwiczenie 3.4
Wyznacz rzut ortogonalny funkcji sin(πx) na przestrzeń wielomianów stopnia nie większego niż 3 na przedziale [−1, 1] i porównaj jego błąd z błędem wielomianu Taylora tego samego stopnia.

### Rozwiązanie 3.4

In [21]:
import sympy as sp
x = sp.symbols("x")
f = sp.sin(sp.pi*x)
il = lambda u, v: sp.integrate(u*v, (x, -1, 1))
baza = [sp.sqrt(sp.Rational(2*n + 1, 2))*sp.legendre(n, x) for n in range(4)]
rzut = sp.expand(sum(il(e, f)*e for e in baza))
taylor = sp.pi*x - (sp.pi*x)**3/6
print(sp.N(rzut, 5))
for nazwa, w in (("rzut", rzut), ("Taylor", taylor)):
    print(nazwa, "norma błędu:", round(float(sp.sqrt(il(f - w, f - w))), 4))

-2.8956*x**3 + 2.6923*x
rzut norma błędu: 0.0937
Taylor norma błędu: 0.8954


### Ćwiczenie 3.5
Dodaj do danych z kodu 3.13 trzeci, czwarty i piąty błąd gruby po tej samej stronie prostej. Przy ilu błędach dopasowanie w normie 1 przestaje odtwarzać prostą?

### Rozwiązanie 3.5

In [22]:
from scipy.optimize import linprog
rng = np.random.default_rng(2)
x = np.linspace(0, 10, 21)
czyste = 1.0 + 0.5*x + rng.normal(scale=0.2, size=x.size)
n = len(x); X = np.stack([np.ones(n), x], axis=1)
def dopasuj_l1(y):
    A = np.block([[X, -np.eye(n)], [-X, -np.eye(n)]])
    return linprog(np.concatenate([[0, 0], np.ones(n)]), A_ub=A,
                   b_ub=np.concatenate([y, -y]),
                   bounds=[(None, None)]*2 + [(0, None)]*n).x[:2]
miejsca = [15, 16, 17, 18, 19, 20, 14, 13, 12, 11, 10]
for ile in (0, 2, 4, 6, 8, 10, 11):
    y = czyste.copy(); y[miejsca[:ile]] += 6.0
    w = dopasuj_l1(y)
    print(f"błędów: {ile:2d}  wyraz wolny {w[0]:6.3f}  nachylenie {w[1]:6.3f}")
# Dopasowanie w normie 1 wytrzymuje wiele błędów, ale nie dowolnie wiele:
# gdy błędne punkty zaczynają dominować po jednej stronie, prosta „przeskakuje”.

błędów:  0  wyraz wolny  0.908  nachylenie  0.510
błędów:  2  wyraz wolny  0.904  nachylenie  0.513
błędów:  4  wyraz wolny  0.888  nachylenie  0.529
błędów:  6  wyraz wolny -0.663  nachylenie  1.283
błędów:  8  wyraz wolny -0.671  nachylenie  1.289
błędów: 10  wyraz wolny -0.259  nachylenie  1.309
błędów: 11  wyraz wolny  0.123  nachylenie  1.295


### Ćwiczenie 3.6
Zwiększaj w kodzie 3.15 liczbę składników sygnału. Przy jakiej liczbie rozwiązanie o najmniejszej normie 1 przestaje pokrywać się z prawdziwymi współczynnikami?

### Rozwiązanie 3.6

In [23]:
from scipy.fft import idct
N = 64
A = np.hstack([np.eye(N), idct(np.eye(N), norm="ortho", axis=0)])
rng = np.random.default_rng(0)
for ile in (4, 8, 12, 16, 20, 24, 28):
    trafienia = 0
    for proba in range(10):
        prawdziwe = np.zeros(2*N)
        prawdziwe[rng.choice(2*N, ile, replace=False)] = rng.normal(size=ile) + 2
        s = A @ prawdziwe
        w = linprog(np.ones(4*N), A_eq=np.hstack([A, -A]), b_eq=s,
                    bounds=[(0, None)]*(4*N)).x
        trafienia += np.allclose(w[:2*N] - w[2*N:], prawdziwe, atol=1e-5)
    print(f"składników: {ile:2d}  poprawnie odtworzone: {trafienia}/10")
# Dopóki składników jest mało w porównaniu z wymiarem przestrzeni,
# norma 1 znajduje je bezbłędnie; powyżej pewnej liczby zaczyna zawodzić.

składników:  4  poprawnie odtworzone: 10/10
składników:  8  poprawnie odtworzone: 10/10
składników: 12  poprawnie odtworzone: 10/10
składników: 16  poprawnie odtworzone: 10/10
składników: 20  poprawnie odtworzone: 9/10
składników: 24  poprawnie odtworzone: 8/10
składników: 28  poprawnie odtworzone: 2/10
